In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import time 

def fetch_article_urls(base_url, page_number):
    """
    Fetch article URLs from a given page.
    """
    article_urls = set()
    url = f"{base_url}/page/{page_number}/"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for link in soup.find_all("a", href=True):
            href = urljoin(url, link['href'])
            if re.match(r'https://www.24-horas.mx/\d{4}/\d{2}/\d{2}/.*/', href):
                article_urls.add(href)
    except requests.RequestException as e:
        print(f"Error fetching URLs from {url}: {e}")
    return list(article_urls)

def clean_main_text(text):
    """
    Remove unwanted phrases from the main text.
    """
    unwanted_phrases = [
        "24 Horas \n\t\t\t\t\tEl Diario Sin Límites\t\t\t\t"
    ]
    for phrase in unwanted_phrases:
        text = text.replace(phrase, "")
    return text.strip()

def extract_article_details(url):
    """
    Extract title, main text, author, and date from an article.
    """

    
    article_data = {}
    try:
        article_data['url'] = url
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title_element = soup.find("h1", class_="entry-title")
        article_data['title'] = title_element.text.strip() if title_element else None

        # Extract and clean main text
        paragraphs = [p.get_text() for p in soup.find_all('p')]
        raw_text = ' '.join(paragraphs)
        article_data['main_text'] = clean_main_text(raw_text)

        # Extract author
        author_element = soup.find("span", class_="author vcard")
        article_data['author'] = author_element.text.strip() if author_element else None

        # Extract date
        date_element = soup.find("time")
        article_data['date'] = date_element['datetime'] if date_element and 'datetime' in date_element.attrs else None

        topic_element = soup.find("a", rel="category tag")
        article_data['topic'] = topic_element.text.strip() if topic_element else None

    except requests.RequestException as e:
        print(f"Error accessing {url}: {e}")
        return None

    return article_data

def scrape_articles(base_url, start_page=1, end_page=10):
    """
    Scrape articles from a given range of pages and save results after every 100 pages.
    """
    df = pd.DataFrame()
    all_articles = []
    for page_number in range(start_page, end_page + 1):
        print(f"Extracting articles from page {page_number}")
        urls = fetch_article_urls(base_url, page_number)
        for url in urls:
            article = extract_article_details(url)
            if article:
                all_articles.append(article)
            
        
        df_new = pd.DataFrame(all_articles)
        df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes
        df = pd.concat([df, df_new])
        all_articles = []
        # Save progress every 100 pages or at the end of the scraping session
        if (page_number % 50 == 0 or page_number == end_page):
            df = df.drop_duplicates(['url']).reset_index(drop=True)
            df.to_parquet(f"../../data/00-newspaper_data/crawler/horas24/articles_cdmx3.parquet", index=False)
            print(f"Data saved'")
                
    
    return(df)

        

# Define the base URL
base_url = "https://www.24-horas.mx/minuto-a-minuto/"
# Example usage: scrape and save the first 10 pages
sc = scrape_articles(base_url, 41101, 55138)


Extracting articles from page 41101
Extracting articles from page 41102
Extracting articles from page 41103
Extracting articles from page 41104
Extracting articles from page 41105
Extracting articles from page 41106
Extracting articles from page 41107
Extracting articles from page 41108
Extracting articles from page 41109
Extracting articles from page 41110
Extracting articles from page 41111
Extracting articles from page 41112
Extracting articles from page 41113
Extracting articles from page 41114
Extracting articles from page 41115
Extracting articles from page 41116
Extracting articles from page 41117
Extracting articles from page 41118
Extracting articles from page 41119
Extracting articles from page 41120
Extracting articles from page 41121
Extracting articles from page 41122
Extracting articles from page 41123
Extracting articles from page 41124
Extracting articles from page 41125
Extracting articles from page 41126
Extracting articles from page 41127
Extracting articles from pag

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_80504/3068579415.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 41445
Extracting articles from page 41446
Extracting articles from page 41447
Extracting articles from page 41448
Extracting articles from page 41449
Extracting articles from page 41450
Data saved'
Extracting articles from page 41451
Extracting articles from page 41452
Extracting articles from page 41453
Extracting articles from page 41454
Extracting articles from page 41455
Extracting articles from page 41456
Extracting articles from page 41457
Extracting articles from page 41458
Extracting articles from page 41459
Extracting articles from page 41460
Extracting articles from page 41461
Extracting articles from page 41462
Extracting articles from page 41463
Extracting articles from page 41464
Extracting articles from page 41465
Extracting articles from page 41466
Extracting articles from page 41467
Extracting articles from page 41468
Extracting articles from page 41469
Extracting articles from page 41470
Extracting articles from page 41471
Extracting artic

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_80504/3068579415.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 41898
Extracting articles from page 41899
Extracting articles from page 41900
Data saved'
Extracting articles from page 41901
Extracting articles from page 41902
Extracting articles from page 41903
Extracting articles from page 41904
Extracting articles from page 41905
Extracting articles from page 41906
Extracting articles from page 41907
Extracting articles from page 41908
Extracting articles from page 41909
Extracting articles from page 41910
Extracting articles from page 41911
Extracting articles from page 41912
Extracting articles from page 41913
Extracting articles from page 41914
Extracting articles from page 41915
Extracting articles from page 41916
Extracting articles from page 41917
Extracting articles from page 41918
Extracting articles from page 41919
Extracting articles from page 41920
Extracting articles from page 41921
Extracting articles from page 41922
Extracting articles from page 41923
Extracting articles from page 41924
Extracting artic

...

Extracting articles from page 48673
Extracting articles from page 48674
Extracting articles from page 48675
Extracting articles from page 48676
Extracting articles from page 48677
Extracting articles from page 48678
Extracting articles from page 48679
Extracting articles from page 48680
Extracting articles from page 48681
Extracting articles from page 48682
Extracting articles from page 48683
Extracting articles from page 48684
Extracting articles from page 48685
Extracting articles from page 48686
Extracting articles from page 48687
Extracting articles from page 48688
Extracting articles from page 48689
Extracting articles from page 48690
Extracting articles from page 48691
Extracting articles from page 48692
Extracting articles from page 48693
Extracting articles from page 48694
Extracting articles from page 48695
Extracting articles from page 48696
Extracting articles from page 48697
Extracting articles from page 48698
Extracting articles from page 48699
Extracting articles from pag

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_80504/3068579415.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 51509
Extracting articles from page 51510
Extracting articles from page 51511
Extracting articles from page 51512
Extracting articles from page 51513
Extracting articles from page 51514
Extracting articles from page 51515
Extracting articles from page 51516
Extracting articles from page 51517
Extracting articles from page 51518
Extracting articles from page 51519
Extracting articles from page 51520
Extracting articles from page 51521
Extracting articles from page 51522
Extracting articles from page 51523
Extracting articles from page 51524
Extracting articles from page 51525
Extracting articles from page 51526
Extracting articles from page 51527
Extracting articles from page 51528
Extracting articles from page 51529
Extracting articles from page 51530
Extracting articles from page 51531
Extracting articles from page 51532
Extracting articles from page 51533
Extracting articles from page 51534
Extracting articles from page 51535
Extracting articles from pag

/var/folders/_8/ftphsw1j58b_rmngtsjm1str0000gp/T/ipykernel_80504/3068579415.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_new['date'] = pd.to_datetime(df_new['date'], utc=True)  # Convert to UTC to handle tz-aware datetimes


Extracting articles from page 52715
Extracting articles from page 52716
Extracting articles from page 52717
Extracting articles from page 52718
Extracting articles from page 52719
Extracting articles from page 52720
Extracting articles from page 52721
Extracting articles from page 52722
Extracting articles from page 52723
Extracting articles from page 52724
Extracting articles from page 52725
Extracting articles from page 52726
Extracting articles from page 52727
Extracting articles from page 52728
Extracting articles from page 52729
Extracting articles from page 52730
Extracting articles from page 52731
Extracting articles from page 52732
Extracting articles from page 52733
Extracting articles from page 52734
Extracting articles from page 52735
Extracting articles from page 52736
Extracting articles from page 52737
Extracting articles from page 52738
Extracting articles from page 52739
Extracting articles from page 52740
Extracting articles from page 52741
Extracting articles from pag

In [ ]:
sc

''